### Exercise: Frequentist Coverage and Loss of Identifiability in NV Center ODMR

**The Physics Context:**
In quantum sensing, Nitrogen-Vacancy (NV) centers in diamond are read out using Optically Detected Magnetic Resonance (ODMR). You sweep a microwave frequency across the resonance and count the emitted photons. At the resonance frequency, the fluorescence drops, creating a "dip" in the spectrum.

To make the numerics manageable, our variable $\nu$ represents the **detuning** (in MHz) from the expected resonance (which is typically ~2.87 GHz). Therefore, a sweep from -5 to +5 means we are sweeping $\pm 5$ MHz around the center.

**The Statistical Model:**
Because we are counting independent photons, the number of observed photons $N_i$ at each discrete frequency point $\nu_i$ follows a Poisson distribution. The expected rate $\lambda(\nu_i)$ is modeled as a constant background $B$ minus a Lorentzian dip of amplitude $A$ (the contrast) and width $\gamma$:
$$\lambda(\nu_i) = B - A \frac{\gamma^2}{(\nu_i - \nu_0)^2 + \gamma^2}$$

The physical meaning and typical values of the parameters for our toy model are:
* **$B$ (Background):** Base photon counts per measurement point (True value = 1000).
* **$A$ (Amplitude/Contrast):** The depth of the dip in photon counts.
* **$\nu_0$ (Center):** The true resonance detuning (True value = 0.0 MHz).
* **$\gamma$ (Linewidth):** The width of the resonance (True value = 1.0 MHz).

**The Goal:**
If the true contrast $A$ is very weak, the signal gets buried in the Poisson noise ($\sqrt{B}$). In this regime, the linewidth $\gamma$ becomes completely unconstrained (a flat likelihood). This is called a "loss of identifiability." Your goal is to study how this affects the 2D Confidence Region of $(A, \gamma)$ and whether Wilks' theorem (which predicts $2\Delta\text{NLL}$ follows a $\chi^2$ distribution) holds at varying signal-to-noise ratios.

**Step-by-Step Instructions:**

1. **The Generative Model & Likelihood:** * Write a Python function `rate_model(nu, B, A, nu0, gamma)` that returns the expected rate.
   * Write a function to generate toy datasets with Poisson fluctuations for the number of photons at 50 discrete points of the frequency detuning $\nu$ between -5 and 5.
   * Write a custom Negative Log-Likelihood (NLL) function for Poisson data.
2. **The Fitter:**
   * Set up an `iminuit.Minuit` object. Set sensible limits (e.g., $B>0, A>0, \gamma>0$).
3. **Checking 2D Coverage (The Core Loop):**
   * Write a function that runs $N_{toys} = 300$ pseudo-experiments for a given true value of $A$.
   * For each toy, perform a fit to find the MLE for the parameters.
   * Then calculate a 2D confidence interval at 68.3% level for the parameters of interest $A$ and $\gamma$.
   * Check if the 2D confidence interval includes the true value of $A$ and $\gamma$ and keep track of the coverage fraction.
4. **Visualizing Wilks' Theorem:**
   * We use a test statistic $t$ based on the profiled likelihood ratio to get the 2D confidence interval of the POIs $A$ and $\gamma$.
   * Plot a histogram of your $t$ values. Overlay the theoretical PDF predicted by Wilks' theorem.
5. **Scanning the Contrast:**
   * Run your loop for three scenarios: a "Strong Signal" ($A=500$), a "Weak Signal" ($A=100$), and a "Very Weak Signal" ($A=20$).
   * Visualize the 2D profile likelihood contours for these three cases. What happens to the contour when the signal is very weak?
   * Compare the coverage fractions and the shapes of the test statistic histograms across the three scenarios.

In [ ]:
 !pip install iminuit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2
from iminuit import Minuit
from scipy.stats import poisson

# ==========================================
# GENERATIVE MODEL & LIKELIHOOD
# ==========================================

def rate_model(nu, B, A, nu0, gamma):
    """Lorentzian dip on a constant background."""
    return B - A * (gamma**2 / ((nu - nu0)**2 + gamma**2))

def generate_toy(nu, B, A, nu0, gamma):
    """Generates a toy dataset with Poisson fluctuations."""
    expected_rate = rate_model(nu, B, A, nu0, gamma)
    # Ensure physical rates (no negative expected photons)
    expected_rate = np.maximum(expected_rate, 1e-6)
    return np.random.poisson(expected_rate)

def create_nll(nu_data, N_data):
    """Creates a custom Poisson NLL function bound to specific data."""
    def nll(B, A, nu0, gamma):
        lam = rate_model(nu_data, B, A, nu0, gamma)
        lam = np.maximum(lam, 1e-6) # Protect against log(0) or negative rates

        # Calculate the negative log-likelihood using scipy
        return -np.sum(poisson.logpmf(N_data, mu=lam))

    return nll

In [ ]:
nu_sweep = np.linspace(-5, 5, 50)
B_true, nu0_true, gamma_true = 1000.0, 0.0, 1.0

N_strong = generate_toy(nu_sweep, B_true, A=500, nu0=nu0_true, gamma=gamma_true)
plt.errorbar(nu_sweep, N_strong, yerr=np.sqrt(N_strong), fmt='ko', markersize=4, label='Toy Data')
plt.ylim(0, 1100)
plt.xlabel("Microwave Detuning $\\nu$ [MHz]")
plt.ylabel("Photon Counts")

In [ ]:
# ==========================================
# VISUALIZING THE MODEL AND TOYS
# ==========================================
nu_sweep = np.linspace(-5, 5, 50)
B_true, nu0_true, gamma_true = 1000.0, 0.0, 1.0

# Generate one toy for each scenario
np.random.seed(42)
N_strong = generate_toy(nu_sweep, B_true, A=500, nu0=nu0_true, gamma=gamma_true)
N_weak   = generate_toy(nu_sweep, B_true, A=100, nu0=nu0_true, gamma=gamma_true)
N_vweak  = generate_toy(nu_sweep, B_true, A=20,  nu0=nu0_true, gamma=gamma_true)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scenarios = [
    (axes[0], N_strong, 500, "Strong Signal (A=500)"),
    (axes[1], N_weak,   100, "Weak Signal (A=100)"),
    (axes[2], N_vweak,  20,  "Very Weak Signal (A=20)")
]

for ax, data, A_val, title in scenarios:
    ax.errorbar(nu_sweep, data, yerr=np.sqrt(data), fmt='ko', markersize=4, label='Toy Data')
    ax.plot(nu_sweep, rate_model(nu_sweep, B_true, A_val, nu0_true, gamma_true),
            'r-', lw=2, label=f'True Rate (A={A_val})')
    ax.set_title(title)
    ax.set_xlabel("Microwave Detuning $\\nu$ [MHz]")
    if ax == axes[0]: ax.set_ylabel("Photon Counts")
    ax.set_ylim(0, 1200)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# VISUALIZING THE 2D CONFIDENCE REGIONS
# ==========================================

def plot_2d_profile_contour(N_data, A_true, title):
    nu_sweep = np.linspace(-5, 5, 50)
    nll_func = create_nll(nu_sweep, N_data)

    # Initialize and run the global unconstrained fit
    m = Minuit(nll_func, B=1000.0, A=A_true, nu0=0.0, gamma=1.0)
    m.limits['B'] = (0, None)
    m.limits['A'] = (0, 1000)
    m.limits['gamma'] = (0.01, 10.0)

    m.migrad()

    plt.figure(figsize=(5, 4))
    if m.valid:
        # draw_mncontour profiles the nuisance parameters automatically!
        # cl=0.683 asks for the 1-sigma 2D region (Delta NLL = 1.15)
        m.draw_mncontour('A', 'gamma', cl=[0.683])

        # Plot the true parameter point
        plt.plot([A_true], [1.0], 'r*', markersize=15)
        plt.title(title, fontsize=14)
        plt.xlabel("Contrast A")
        plt.ylabel(r"Linewidth $\gamma$")
    else:
        print(f"Fit failed for {title}")
    plt.show()


In [ ]:
# Generate one strong and one weak toy
np.random.seed(42) # Set seed for reproducible contours
N_strong = generate_toy(np.linspace(-5, 5, 50), 1000.0, 300, 0.0, 1.0)
N_weak   = generate_toy(np.linspace(-5, 5, 50), 1000.0, 100, 0.0, 1.0)
N_vweak  = generate_toy(np.linspace(-5, 5, 50), 1000.0, 20, 0.0, 1.0)

# Plot both
plot_2d_profile_contour(N_strong, 300, "2D 68.3% CL Region (Strong Signal: A=300)")
plot_2d_profile_contour(N_weak, 100, "2D 68.3% CL Region (Weak Signal: A=100)")
plot_2d_profile_contour(N_vweak, 20, "2D 68.3% CL Region (Very Weak Signal: A=20)")
#@title { vertical-output: true}

In [ ]:
# ==========================================
# 2, 3 & 4: THE CORE LOOP & WILKS' THEOREM
# ==========================================

def run_coverage_study(A_true, B_true=1000.0, nu0_true=0.0, gamma_true=1.0, N_toys=300):
    nu_sweep = np.linspace(-5, 5, 50)

    # Critical value for 68.3% coverage in 2D (chi2 with df=2)
    # scipy.stats.chi2(2).cdf(2.30) is approx 0.6827
    t_critical_2D = chi2(2).ppf(0.6827)

    t_values = []
    covered_count = 0

    for i in range(N_toys):
        # 1. Generate Data
        N_obs = generate_toy(nu_sweep, B_true, A_true, nu0_true, gamma_true)
        nll_func = create_nll(nu_sweep, N_obs)

        # 2. Unconstrained Fit (Global Minimum)
        m_unconstrained = Minuit(nll_func, B=B_true, A=A_true, nu0=nu0_true, gamma=gamma_true)
        m_unconstrained.limits['B'] = (0, None)
        m_unconstrained.limits['A'] = (0, B_true) # Contrast cannot exceed background
        m_unconstrained.limits['gamma'] = (0.01, 10.0) # Width must be strictly positive

        m_unconstrained.migrad()
        if not m_unconstrained.valid:
            continue # Skip failed fits

        NLL_min = m_unconstrained.fval

        # 3. Constrained Fit (Fixing POIs A and gamma to True Values)
        # We profile the nuisance parameters B and nu0
        m_constrained = Minuit(nll_func, B=B_true, A=A_true, nu0=nu0_true, gamma=gamma_true)
        m_constrained.limits['B'] = (0, None)

        # FIX the parameters of interest
        m_constrained.fixed['A'] = True
        m_constrained.fixed['gamma'] = True

        m_constrained.migrad()
        NLL_true = m_constrained.fval

        # 4. Calculate Test Statistic
        # t = 2 * Delta NLL
        t = 2.0 * (NLL_true - NLL_min)

        # Protect against tiny negative values due to numerical precision
        t = max(t, 0.0)
        t_values.append(t)

        # 5. Check Coverage
        if t <= t_critical_2D:
            covered_count += 1

    coverage_fraction = covered_count / len(t_values)
    return np.array(t_values), coverage_fraction

In [ ]:
# ==========================================
# SCANNING CONTRAST AND PLOTTING HISTOGRAMS
# ==========================================
print("Running Strong Signal Toys (A=500)...")
t_strong, cov_strong = run_coverage_study(A_true=500, N_toys=1000)

print("Running Weak Signal Toys (A=100)...")
t_weak, cov_weak = run_coverage_study(A_true=100, N_toys=1000)

print("Running Very Weak Signal Toys (A=20)...")
t_vweak, cov_vweak = run_coverage_study(A_true=20, N_toys=1000)

# Theoretical PDF for Wilks' theorem (chi-square with 2 degrees of freedom)
x_vals = np.linspace(0, 15, 200)
chi2_pdf = chi2(2).pdf(x_vals)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

results = [
    (axes[0], t_strong, cov_strong, "Strong Signal (A=500)", 'blue'),
    (axes[1], t_weak,   cov_weak,   "Weak Signal (A=100)", 'green'),
    (axes[2], t_vweak,  cov_vweak,  "Very Weak Signal (A=20)", 'orange')
]

for ax, t_vals, cov, title, color in results:
    ax.hist(t_vals, bins=30, range=(0,15), density=True, alpha=0.6, color=color, label='Toys')
    ax.plot(x_vals, chi2_pdf, 'k--', lw=2, label=r'Wilks: $\chi^2(df=2)$')
    ax.axvline(chi2(2).ppf(0.6827), color='red', linestyle=':', lw=2, label='68.3% Limit (2.30)')
    ax.set_title(f"{title}\nCoverage = {cov*100:.1f}%")
    ax.set_xlabel(r"Test Statistic $t = 2\Delta$NLL")
    if ax == axes[0]: ax.set_ylabel("Probability Density")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# PROVING THE LOOK-ELSEWHERE EFFECT (2D)
# ==========================================

def run_lee_proof_study_2d(A_true=20, B_true=1000.0, nu0_true=0.0, gamma_true=1.0, N_toys=500):
    nu_sweep = np.linspace(-5, 5, 50)

    # Critical value for 68.3% coverage in 2D (chi2 with df=2)
    # scipy.stats.chi2(2).ppf(0.6827) is approx 2.30
    t_critical_2D = chi2(2).ppf(0.6827)

    t_values = []
    covered_count = 0

    for i in range(N_toys):
        N_obs = generate_toy(nu_sweep, B_true, A_true, nu0_true, gamma_true)
        nll_func = create_nll(nu_sweep, N_obs)

        # 1. Unconstrained Fit (But we freeze the roaming position!)
        m_unconstrained = Minuit(nll_func, B=B_true, A=A_true, nu0=nu0_true, gamma=gamma_true)
        m_unconstrained.limits['B'] = (0, None)
        m_unconstrained.limits['A'] = (0, B_true)
        m_unconstrained.limits['gamma'] = (0.01, 10.0)

        # FREEZE nu0 to prevent scanning the Look-Elsewhere Effect
        m_unconstrained.fixed['nu0'] = True

        m_unconstrained.migrad()
        if not m_unconstrained.valid: continue
        NLL_min = m_unconstrained.fval

        # 2. Constrained Fit (Fix POIs A and gamma to truth to test them)
        m_constrained = Minuit(nll_func, B=B_true, A=A_true, nu0=nu0_true, gamma=gamma_true)
        m_constrained.limits['B'] = (0, None)

        # nu0 is fixed here as well to maintain consistency
        m_constrained.fixed['nu0'] = True

        # FIX the POIs
        m_constrained.fixed['A'] = True
        m_constrained.fixed['gamma'] = True

        m_constrained.migrad()
        NLL_true = m_constrained.fval

        t = 2.0 * (NLL_true - NLL_min)
        t = max(t, 0.0)
        t_values.append(t)

        if t <= t_critical_2D:
            covered_count += 1

    coverage_fraction = covered_count / len(t_values)
    return np.array(t_values), coverage_fraction


In [ ]:
print("Running 2D Look-Elsewhere Proof (A=20 with Fixed nu0)...")
t_lee_2d, cov_lee_2d = run_lee_proof_study_2d(A_true=20, N_toys=1000)

# Plot the results
x_vals_2d = np.linspace(0, 15, 200)
chi2_2d_pdf = chi2(2).pdf(x_vals_2d)

plt.figure(figsize=(7, 5))
plt.hist(t_lee_2d, bins=30, range=(0,15), density=True, alpha=0.6, color='purple', label='Toys (Fixed $\\nu_0$)')
plt.plot(x_vals_2d, chi2_2d_pdf, 'k--', lw=2, label=r'Wilks: $\chi^2(df=2)$')
plt.axvline(chi2(2).ppf(0.6827), color='red', linestyle=':', lw=2, label='68.3% Critical Limit (2.30)')

plt.title(f"2D Coverage for A=20 with Fixed Position (Isolating LEE)\nCoverage Restored = {cov_lee_2d*100:.1f}%", fontsize=14)
plt.xlabel(r"Test Statistic $t = 2\Delta$NLL")
plt.ylabel("Probability Density")
plt.legend()
plt.tight_layout()
plt.show()
#@title { vertical-output: true}